In [1]:
import re
from docling.document_converter import DocumentConverter,PdfFormatOption
from docling.datamodel.pipeline_options import PdfPipelineOptions
from docling.backend.pypdfium2_backend import PyPdfiumDocumentBackend
from docling.datamodel.base_models import InputFormat
pipeline_options = PdfPipelineOptions()
pipeline_options.do_ocr = False
pipeline_options.do_table_structure = True
pipeline_options.table_structure_options.do_cell_matching = False

In [6]:
def pdf2md_single(source) :
    converter = DocumentConverter(format_options={ InputFormat.PDF: PdfFormatOption(
                pipeline_options=pipeline_options, backend=PyPdfiumDocumentBackend
            )
        }
    )
    result = converter.convert(source)
    return result.document.export_to_markdown()


file = open('pdf2md.md','w',encoding='utf-8')
file.write(pdf2md_single("../unique_sxolika_pdf/paper_16.pdf"))

Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

37724

In [7]:
def paragraph_maker(text,maxpadding = 2) :
    lines = text.splitlines()
    paragraphs = []
    emptyline = 0
    paragraph = ''
    for i,line in enumerate(lines) :
        if i == len(lines) - 1 :
            paragraph = paragraph + line
            paragraphs.append(paragraph)
            continue
        if line == '' :
            emptyline = emptyline + 1
            if emptyline == maxpadding :
                if paragraph != '' :
                    paragraphs.append(paragraph)
                emptyline = 0
                paragraph = ''
        else :
            paragraph = paragraph + line + '\n'
            emptyline = 0
            if i == len(lines) - 1 :
                paragraphs.append(paragraph)
    return paragraphs

def paragraph_merger(paragraphs,min_par_size,threshold) :
    del_index = []
    for i,paragraph in enumerate(paragraphs) :
        if len(paragraph) < min_par_size :
            if paragraph != paragraphs[-1] :
                if len(paragraphs[i+1]) > threshold :
                    paragraphs[i+1] = paragraph + '\n MERGE \n' + paragraphs[i+1]
                    del_index.append(i)
    for index in sorted(del_index, reverse=True) :
        del paragraphs[index]
    return paragraphs

def paragraph_not_char_end(paragraph,chars) :
    if not paragraph.endswith(chars) :
        paragraph = '[Out:Paragraph does not end with specified char]' + paragraph
    return paragraph

def all_paragraph_not_char_end(paragraphs,chars) :
    newparagraphs = []
    for paragraph in paragraphs :
        newparagraphs.append(paragraph_not_char_end(paragraph,chars))
    return newparagraphs

def print_text(paragraphs) :
    for paragraph in paragraphs :
        print(paragraph)
        print('\n\n\n')

def write_text(paragraphs,file) :
    text = '[Paragraph Begin]'
    for paragraph in paragraphs :
        text = text + paragraph + '[Paragraph End]\n\n\n[Paragraph Begin]'
    file.write(text) 

In [ ]:
file = open('pdf2md.md','r',encoding='utf-8')
paragraphs = paragraph_maker(file.read(),maxpadding=1)
paragraphs = paragraph_merger(paragraphs,50,10)
endings = ('.\n','?\n','!\n',';\n','.\n',';\n')
paragraphs = all_paragraph_not_char_end(paragraphs,endings)
file_out = open('test_paper_16.md','w',encoding='utf-8')
write_text(paragraphs,file_out)